# Part 2: Custom Inference & Attack Notebook

This notebook uses the custom `Model` class from your original notebook to evaluate robustness.

In [ ]:
!pip install timm torchinfo matplotlib

In [ ]:
import torch
import torch.nn as nn
import timm

class Model(nn.Module):
    def __init__(self, model_name='resnetv2_50x1_bit', num_classes=10):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True)
        self.model.stem.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.stem.pool = nn.Identity()

        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.stem.conv.parameters():
            param.requires_grad = True
        for param in self.model.stages[3].parameters():
            param.requires_grad = True

        in_channels = self.model.head.fc.in_channels 
        self.model.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(in_channels, num_classes)
        )
        for param in self.model.head.parameters():
            param.requires_grad = True
    def forward(self, x):
        return self.model(x)

In [ ]:
%%writefile attacks.py
import torch
import torch.nn.functional as F

class AdversarialAttack:
    def __init__(self, model, epsilon=0.1, alpha=0.01, iters=10):
        self.model = model
        self.epsilon = epsilon
        self.alpha = alpha
        self.iters = iters
    def fgsm(self, images, labels):
        images = images.clone().detach().to(images.device)
        labels = labels.clone().detach().to(images.device)
        images.requires_grad = True
        outputs = self.model(images)
        loss = F.cross_entropy(outputs, labels)
        self.model.zero_grad()
        loss.backward()
        perturbed_image = images + self.epsilon * images.grad.data.sign()
        return torch.clamp(perturbed_image, -3, 3)
    def pgd(self, images, labels):
        ori_images = images.clone().detach().to(images.device)
        images = images.clone().detach().to(images.device)
        for i in range(self.iters):
            images.requires_grad = True
            outputs = self.model(images)
            loss = F.cross_entropy(outputs, labels)
            self.model.zero_grad()
            loss.backward()
            adv_images = images + self.alpha * images.grad.data.sign()
            eta = torch.clamp(adv_images - ori_images, min=-self.epsilon, max=self.epsilon)
            images = (ori_images + eta).detach()
        return images

In [ ]:
%%writefile inference.py
import torch
import torch.nn as nn

def bit_depth_reduction(images, bits=3):
    levels = 2 ** bits
    return torch.round(images * (levels - 1)) / (levels - 1)

def smoothing_defense(images, kernel_size=3):
    padding = kernel_size // 2
    avg_pool = nn.AvgPool2d(kernel_size=kernel_size, stride=1, padding=padding)
    return avg_pool(images)

class RobustInference:
    def __init__(self, model, device='cpu'):
        self.model = model.to(device)
        self.model.eval()
    def predict(self, images, defense=None):
        if defense == 'bit_depth':
            images = bit_depth_reduction(images)
        elif defense == 'smoothing':
            images = smoothing_defense(images)
        with torch.no_grad():
            outputs = self.model(images)
            _, predicted = torch.max(outputs, 1)
        return predicted, outputs

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from attacks import AdversarialAttack
from inference import RobustInference

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(num_classes=10).to(DEVICE)
try:
    model.load_state_dict(torch.load("resnet_bit_cifar10.pth", map_location=DEVICE))
except: pass

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
test_loader = DataLoader(datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test), batch_size=32, shuffle=True)

attacker = AdversarialAttack(model, epsilon=8/255.0)
infer = RobustInference(model, device=DEVICE)

data, target = next(iter(test_loader))
data, target = data.to(DEVICE), target.to(DEVICE)
data_adv = attacker.pgd(data, target)

pred_clean, _ = infer.predict(data)
pred_adv, _ = infer.predict(data_adv)
print(f"Clean Acc: {pred_clean.eq(target).sum().item()/32:.2%}")
print(f"Adv Acc: {pred_adv.eq(target).sum().item()/32:.2%}")